# NeuroMM-2026 — Pipeline End-to-End

**Solution complète pour le challenge NeuroMM-2026 : détection multimodale de crises d'épilepsie**

Ce notebook couvre l'intégralité du pipeline :

| # | Section | Description |
|---|---|---|
| 0 | [Environnement](#0-environnement) | Installation, GPU check |
| 1 | [Données](#1-téléchargement-des-données) | Téléchargement HuggingFace, validation, exploration |
| 2 | [Configuration](#2-configuration) | YAML base + overrides par track |
| 3 | [Architecture](#3-architecture-du-modèle) | Y-Architecture EEGMamba, comptage de paramètres |
| 4 | [Smoke Test](#4-smoke-test) | Validation rapide sur 10% des données |
| 5 | [Pré-entraînement](#5-pré-entraînement-multi-tâche) | Backbone multi-tâche (binaire + reconstruction + label_type) |
| 6 | [Track 1](#6-track-1--eeg-only) | Fine-tuning détection binaire EEG only |
| 7 | [Track 2](#7-track-2--eeg--vidéo) | Fine-tuning avec Uncertainty-aware Gating vidéo |
| 8 | [Track 3](#8-track-3--localisation-5-classes) | Fine-tuning localisation épileptogène 5-classes |
| 9 | [Évaluation](#9-évaluation--calibration) | AUPRC, Weighted-F1, calibration isotonique |
| 10 | [Soumission](#10-inférence-et-soumission) | Inférence candidat → ZIP Codabench |
| 11 | [Tests](#11-tests-unitaires) | Suite pytest complète |

---

**Dataset** : 25 426 fenêtres EEG (29 canaux × 2000 points à 500 Hz) + features vidéo pré-extraites de 7 backbones  
**Métriques** : AUPRC (Track 1/2), Weighted-F1 (Track 3)  
**Leaderboard** : https://2026.neuromm.org

---
## 0. Environnement

### Installation des dépendances

In [ ]:
# Dépendances core
!pip install -q torch>=2.1.0 numpy>=1.24.0 pandas>=2.0.0 scikit-learn>=1.3.0 \
             scipy>=1.11.0 pyyaml>=6.0 tqdm>=4.65.0 pytest>=7.4.0 \
             huggingface_hub>=0.21.0

# Mamba2 CUDA (recommandé sur GPU CUDA 11.8+, ~4-8x plus rapide)
# !pip install -q mamba-ssm causal-conv1d

# LoRA fine-tuning (uniquement si --foundation_ckpt est utilisé)
# !pip install -q peft

# Experiment tracking
# !pip install -q wandb

print("Installation terminée.")

In [ ]:
import sys, os
from pathlib import Path
import torch
import numpy as np

# ── Racine du projet ──────────────────────────────────────────────────────
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── GPU / CPU check ───────────────────────────────────────────────────────
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA dispo  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU         : {gpu.name}")
    print(f"VRAM        : {gpu.total_memory / 1024**3:.1f} GB")
    DEVICE = torch.device("cuda")
    BATCH_SIZE = 256 if gpu.total_memory > 20 * 1024**3 else 128  # L4 vs T4
else:
    print("⚠  CPU uniquement — entraînement lent (smoke test recommandé)")
    DEVICE = torch.device("cpu")
    BATCH_SIZE = 16

print(f"\nDevice      : {DEVICE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Projet root : {PROJECT_ROOT}")

---
## 1. Téléchargement des données

Le dataset nécessite un accès HuggingFace :  
**Demander l'accès** → https://huggingface.co/datasets/NeuroMM/NeuroMM-2026

```
NeuroMM-2026/
├── annotations/neuromm2026_train_val.csv   ← 25 426 lignes
├── archives/
│   ├── eeg.tar                             ← (29, 2000) float32
│   ├── video_videomae-large.tar            ← (1, 1024)
│   ├── video_dinov2-large.tar              ← (8, 1024)
│   └── ...  (5 autres backbones)
└── candidate/                              ← set test (sans labels)
```

In [ ]:
# Configurer le token HuggingFace
# Option A — variable d'environnement (recommandé)
#   export HF_TOKEN=hf_XXXX
# Option B — directement ici (ne pas committer)

HF_TOKEN = os.getenv("HF_TOKEN", "")

if HF_TOKEN:
    print(f"Token HF trouvé : {HF_TOKEN[:8]}...")
else:
    print("⚠  HF_TOKEN non défini.")
    print("   Définir avec : export HF_TOKEN=hf_XXXX")
    print("   Ou remplacer la chaîne vide ci-dessus par votre token.")

In [ ]:
# ── Téléchargement + extraction + validation + index ─────────────────────
# Cette cellule exécute les 5 étapes de prepare_data.py :
#   1. Download  — archives depuis HuggingFace
#   2. Extract   — décompression dans processed/features/
#   3. Validate  — vérification des shapes
#   4. Index     — flags de disponibilité par backbone
#   5. Subset    — sous-ensemble smoke 10% patient-stratifié

!python scripts/prepare_data.py \
    --hf_token $HF_TOKEN \
    --backbones videomae-large dinov2-large

# Pour tous les backbones + le set candidat :
# !python scripts/prepare_data.py --hf_token $HF_TOKEN --backbones all --include_candidate

# Si les archives sont déjà sur disque (re-validation uniquement) :
# !python scripts/prepare_data.py --skip_download --skip_extract

In [ ]:
import pandas as pd

# ── Chargement du manifest ────────────────────────────────────────────────
df = pd.read_csv("annotations/neuromm2026_train_val.csv")
print(f"Total samples : {len(df):,}")
print(f"\nDistribution split :")
print(df["split"].value_counts().to_string())
print(f"\nDistribution labels (binaire) :")
print(df["label"].value_counts().to_string())
print(f"\nDistribution label_type (multi-classe) :")
print(df["label_type"].value_counts().sort_index().to_string())
print(f"\nNombre de sujets : {df['subject_id'].nunique()}")

In [ ]:
# ── Inspection d'un échantillon EEG ──────────────────────────────────────
sample_id = df.iloc[0]["sample_id"]
eeg = np.load(f"processed/features/eeg/{sample_id}.npy")
print(f"sample_id : {sample_id}")
print(f"shape     : {eeg.shape}   ← (29 canaux, 2000 timesteps @ 500 Hz = 4s)")
print(f"dtype     : {eeg.dtype}")
print(f"min/max   : {eeg.min():.4f} / {eeg.max():.4f}")

# Inspection d'un vecteur vidéo DINOv2-Large
try:
    vid = np.load(f"processed/features/video/dinov2-large/{sample_id}.npy")
    print(f"\nDINOv2-Large video features :")
    print(f"shape : {vid.shape}   ← (8 frames, 1024 dim)")
    print(f"dtype : {vid.dtype}")
except FileNotFoundError:
    print("\nFeatures vidéo non disponibles (téléchargement nécessaire).")

In [ ]:
# ── Visualisation d'un signal EEG ─────────────────────────────────────────
try:
    import matplotlib.pyplot as plt

    # Chercher un vrai positif (spike)
    pos_id = df[df["label"] == 1].iloc[0]["sample_id"]
    neg_id = df[df["label"] == 0].iloc[0]["sample_id"]

    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    t = np.arange(2000) / 500.0  # temps en secondes

    for ax, sid, title, color in zip(
        axes,
        [pos_id, neg_id],
        ["Positif (spike/seizure)", "Négatif (fond de carte)"],
        ["crimson", "steelblue"],
    ):
        eeg = np.load(f"processed/features/eeg/{sid}.npy")
        offset = np.arange(29) * 5  # décalage vertical par canal
        for c in range(29):
            ax.plot(t, eeg[c] + offset[c], color=color, linewidth=0.5, alpha=0.7)
        ax.set_title(f"{title}  (id={sid})", fontsize=11)
        ax.set_xlabel("Temps (s)")
        ax.set_ylabel("Canaux EEG")
        ax.set_yticks(offset[::4])
        ax.set_yticklabels([f"Ch{c+1}" for c in range(0, 29, 4)])

    plt.tight_layout()
    plt.savefig("logs/eeg_visualization.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Visualisation sauvegardée dans logs/eeg_visualization.png")
except ImportError:
    print("matplotlib non installé — pip install matplotlib pour la visualisation")
except FileNotFoundError:
    print("Données non disponibles — lancer la cellule de téléchargement d'abord.")

---
## 2. Configuration

Les configurations YAML suivent un système d'héritage :  
`base.yaml` → `track1.yaml` / `track2.yaml` / `track3.yaml`

Chaque track override uniquement ce qu'il change (training, loss, video).

In [ ]:
from src.utils.config import load_config
import pprint

cfg_base   = load_config("configs/base.yaml")
cfg_track1 = load_config("configs/track1.yaml")
cfg_track2 = load_config("configs/track2.yaml")
cfg_track3 = load_config("configs/track3.yaml")

print("=== BACKBONE (partagé par tous les tracks) ===")
pprint.pprint(cfg_base["backbone"])

print("\n=== PRETRAINING ===")
pprint.pprint(cfg_base["pretrain"])

print("\n=== AUGMENTATIONS ===")
pprint.pprint(cfg_base["augmentation"])

In [ ]:
print("=== TRACK 1 (EEG only, AUPRC) ===")
print(f"  Track      : {cfg_track1.get('track')}")
print(f"  Use video  : {cfg_track1.get('use_video')}")
print(f"  Loss       : {cfg_track1.get('loss', {}).get('primary')}")
print(f"  Epochs     : {cfg_track1.get('training', {}).get('epochs')}")
print(f"  LR head    : {cfg_track1.get('training', {}).get('lr')}")
print(f"  LR backbone: {cfg_track1.get('training', {}).get('backbone_lr')}")

print("\n=== TRACK 2 (EEG + Video, AUPRC) ===")
print(f"  Active backbones : {cfg_track2.get('video', {}).get('active_backbones')}")
print(f"  Entropy threshold: {cfg_track2.get('gating', {}).get('entropy_threshold')}")
print(f"  MC dropout passes: {cfg_track2.get('gating', {}).get('mc_dropout_passes')}")

print("\n=== TRACK 3 (EEG + Video positifs, Weighted-F1) ===")
print(f"  n_classes          : {cfg_track3.get('n_classes')}")
print(f"  Label smoothing    : {cfg_track3.get('loss', {}).get('label_smoothing')}")
print(f"  Active backbone    : {cfg_track3.get('video', {}).get('active_backbones', ['?'])[0]}")

---
## 3. Architecture du modèle

```
EEG (B, 29, 2000)
      │
      ▼
  LocalCNN        ← per-canal, morphologie spike 20-70ms
      │              (B, 29, 64)
  DynamicGAT      ← adjacence inter-canal apprise dynamiquement
      │              (B, 29, 128)
  EEGMamba        ← Mamba2 SSM, O(n), 4 couches, d_model=256
      │              (B, 256)
      ├── Track1Head  → binary logit       → AUPRC
      ├── Track2Head  → gate + cross-attn  → AUPRC
      └── Track3Head  → 5-class logits     → Weighted-F1
```

In [ ]:
from src.models import build_model_from_config
from src.models.backbone import BackboneConfig

# Construire le modèle complet depuis la config base
model = build_model_from_config(cfg_base)
model = model.to(DEVICE)

counts = model.count_parameters()
print("=== Paramètres du modèle ===")
for k, v in counts.items():
    bar = "█" * (v // 100_000)
    print(f"  {k:<20} : {v:>10,}  {bar}")

print(f"\nTotal entraînable : {counts['total']:,}")

In [ ]:
# ── Test du forward pass sur tenseurs synthétiques ───────────────────────
B, C, T = 4, 29, 2000
dummy_eeg = torch.randn(B, C, T, device=DEVICE)
dummy_vid = {
    "videomae-large": torch.randn(B, 1, 1024, device=DEVICE),
    "dinov2-large":   torch.randn(B, 8, 1024, device=DEVICE),
}

with torch.no_grad():
    # Track 1
    out1 = model(dummy_eeg, mode="track1")
    print(f"Track 1 logit  : {out1['logit'].shape}  ← (B,)")

    # Track 2
    out2 = model(dummy_eeg, video_features=dummy_vid, mode="track2")
    print(f"Track 2 logit  : {out2['logit'].shape}  ← (B,)")
    print(f"Track 2 gate   : {out2['gate'].shape}   ← (B,) ∈ [0,1]")
    print(f"  Gate moyenne : {out2['gate'].mean().item():.3f}  (0=EEG certain, 1=EEG incertain)")

    # Track 3
    out3 = model(dummy_eeg, video_features=dummy_vid, mode="track3")
    print(f"Track 3 logits : {out3['logit'].shape}  ← (B, 5)")

    # Pré-entraînement
    masked = dummy_eeg.clone()
    masked[:, :, :400] = 0.0
    out_pt = model(dummy_eeg, mode="pretrain", masked_eeg=masked)
    print(f"\nPretrain binary logit    : {out_pt['binary_logit'].shape}")
    print(f"Pretrain labeltype logit : {out_pt['labeltype_logit'].shape}")
    print(f"Pretrain recon           : {out_pt['recon'].shape}")

print("\n✓ Forward pass réussi sur tous les modes.")

In [ ]:
# ── Architecture détaillée des sous-modules ──────────────────────────────
def describe_module(module, name="", indent=0):
    """Affichage léger de l'arbre de modules."""
    n_params = sum(p.numel() for p in module.parameters() if p.requires_grad)
    prefix = "  " * indent
    print(f"{prefix}{name} ({module.__class__.__name__}) — {n_params:,} params")
    if indent < 2:
        for child_name, child in module.named_children():
            describe_module(child, child_name, indent + 1)

print("=== Architecture NeuroMMModel ===")
describe_module(model, "NeuroMMModel")

---
## 4. Smoke Test

**Toujours exécuter avant un entraînement complet.** Le smoke test utilise 10% des patients (patient-disjoint, stratifié) et valide :
- Chargement des données
- Forward + backward pass
- Calcul de loss et métriques
- Sauvegarde des checkpoints

Durée estimée : **2-5 min sur GPU** (T4/L4), ~30 min sur CPU.

In [ ]:
# ── Génération du sous-ensemble smoke (si non existant) ──────────────────
from pathlib import Path

smoke_csv = Path("annotations/neuromm2026_smoke_10pct.csv")
if not smoke_csv.exists():
    print("Génération du sous-ensemble smoke 10%...")
    !python scripts/prepare_data.py --skip_download --skip_extract --smoke_pct 0.10
else:
    df_smoke = pd.read_csv(smoke_csv)
    print(f"Smoke subset existant : {len(df_smoke)} samples")
    print(df_smoke["split"].value_counts().to_string())

In [ ]:
# ── Smoke test : 2 epochs de pré-entraînement ────────────────────────────
!python scripts/pretrain.py \
    --config configs/base.yaml \
    --annotations annotations/neuromm2026_smoke_10pct.csv \
    --max_epochs 2 \
    --batch_size {BATCH_SIZE} \
    --num_workers 4

# Vérifier la présence du checkpoint
ckpt = Path("checkpoints/pretrain/best_backbone.pt")
if ckpt.exists():
    print(f"\n✓ Checkpoint créé : {ckpt}  ({ckpt.stat().st_size / 1024**2:.1f} MB)")
else:
    print("\n✗ Checkpoint non trouvé — vérifier les logs ci-dessus")

---
## 5. Pré-entraînement multi-tâche

Le backbone est pré-entraîné avec **trois objectifs simultanés** :

| Objectif | Poids | Rôle |
|---|---|---|
| Détection binaire (FocalPoly) | λ=1.0 | Discriminer spike/non-spike |
| Reconstruction masquée (MSE) | λ=0.3 | Représentation temporelle robuste |
| Classification label_type (CE) | λ=0.5 | Enrichit le backbone pour Track 3 |

**GPU recommandé :**
- T4 (16 GB) : `--batch_size 128` → ~20 min pour 60 epochs
- L4 (24 GB) : `--batch_size 256` → ~10 min
- CPU : `--batch_size 16` → ~15h (utiliser smoke subset)

In [ ]:
# ── Pré-entraînement complet ─────────────────────────────────────────────
!python scripts/pretrain.py \
    --config configs/base.yaml \
    --batch_size {BATCH_SIZE} \
    --num_workers 4

# Option avec foundation model (LoRA) :
# !python scripts/pretrain.py \
#     --config configs/base.yaml \
#     --foundation_ckpt pretrained/eegmamba_base.pt \
#     --batch_size {BATCH_SIZE}

In [ ]:
# ── Inspection du checkpoint sauvegardé ──────────────────────────────────
ckpt_path = Path("checkpoints/pretrain/best_backbone.pt")
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu")
    print(f"Checkpoint : {ckpt_path}")
    print(f"  Epoch    : {ckpt.get('epoch', 'N/A')}")
    print(f"  Best loss: {ckpt.get('loss', 'N/A'):.4f}")
    print(f"  Clés     : {list(ckpt.keys())}")
    n_keys = len(ckpt.get('backbone_state', {}))
    print(f"  Tenseurs backbone : {n_keys}")
else:
    print("Checkpoint non trouvé — lancer le pré-entraînement d'abord.")

---
## 6. Track 1 — EEG only

**Tâche** : détection binaire spike/non-spike sur signal EEG seul.  
**Métrique** : AUPRC (Area Under Precision-Recall Curve).  
**Stratégie** :
- 5-fold CV patient-disjoint (GroupKFold sur `subject_id`)
- RobustScaler fitté sur fold-train uniquement
- WeightedRandomSampler (cap neg:pos ≤ 4)
- Augmentations : SpecAugment + GaussianNoise
- FocalPolyLoss (γ=2, α=0.75, ε=1.0)
- Calibration isotonique sur les scores OOF

In [ ]:
# ── Fine-tuning Track 1 — 5-fold CV ──────────────────────────────────────
BACKBONE_CKPT = "checkpoints/pretrain/best_backbone.pt"

!python scripts/train_track.py \
    --track 1 \
    --config configs/track1.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
# ── Évaluation Track 1 sur le split validation officiel ──────────────────
from src.models import build_model_from_config
from src.data.dataset import build_dataloader
from src.data.preprocessing import load_scalers
from src.evaluation.metrics import compute_auprc, compute_auroc
import pickle

device = DEVICE
df_full = pd.read_csv("annotations/neuromm2026_train_val.csv")
df_val  = df_full[df_full["split"] == "val"].reset_index(drop=True)

# Charger les scalers du fold 0 (ou full-train si disponibles)
scaler_path = Path("checkpoints/track1/scalers_fold0.pkl")
if not scaler_path.exists():
    print(f"Scalers non trouvés : {scaler_path}")
else:
    scalers = load_scalers(scaler_path)
    val_loader = build_dataloader(
        df_val,
        eeg_dir=Path("processed/features/eeg"),
        scalers=scalers,
        track=1,
        batch_size=128,
        augment=False,
        shuffle=False,
        num_workers=4,
    )

    # Ensemble des 5 folds
    fold_scores = []
    for fold_idx in range(5):
        ckpt_path = Path(f"checkpoints/track1/fold{fold_idx}.pt")
        if not ckpt_path.exists():
            continue
        model = build_model_from_config(cfg_track1).to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        model.eval()

        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                logit = model(batch["eeg"].to(device), mode="track1")["logit"]
                all_logits.append(torch.sigmoid(logit).cpu().numpy())
                all_labels.append(batch["label"].numpy())

        s = np.concatenate(all_logits)
        t = np.concatenate(all_labels)
        fold_scores.append(s)
        print(f"  Fold {fold_idx} AUPRC = {compute_auprc(t, s):.4f}")

    if fold_scores:
        ensemble_s = np.mean(fold_scores, axis=0)

        # Calibration isotonique
        calib_path = Path("checkpoints/track1/calibrator.pkl")
        if calib_path.exists():
            with open(calib_path, "rb") as f:
                calib = pickle.load(f)
            ensemble_s_cal = calib.transform(ensemble_s)
            print(f"\nEnsemble AUPRC (brut)     : {compute_auprc(t, ensemble_s):.4f}")
            print(f"Ensemble AUPRC (calibré)  : {compute_auprc(t, ensemble_s_cal):.4f}")
        else:
            print(f"\nEnsemble AUPRC            : {compute_auprc(t, ensemble_s):.4f}")
            print(f"Ensemble AUROC            : {compute_auroc(t, ensemble_s):.4f}")

---
## 7. Track 2 — EEG + Vidéo

**Nouveauté** : Uncertainty-aware Gating  
Le modèle utilise **MC-Dropout** pour estimer l'entropie prédictive de l'EEG.  
- Si entropie < seuil → EEG confiant → vidéo supprimée (gate ≈ 0)  
- Si entropie > seuil → EEG incertain → cross-attention vidéo activée (gate ≈ 1)

Ceci mime la pratique clinique : la caméra est consultée quand l'EEG est ambigu (artefacts de mouvement).

In [ ]:
# ── Fine-tuning Track 2 ───────────────────────────────────────────────────
!python scripts/train_track.py \
    --track 2 \
    --config configs/track2.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
# ── Analyse de l'uncertainty gate ────────────────────────────────────────
# Visualiser les valeurs de gate sur quelques exemples de validation

ckpt_path = Path("checkpoints/track2/fold0.pt")
scaler_path = Path("checkpoints/track2/scalers_fold0.pkl")

if ckpt_path.exists() and scaler_path.exists():
    scalers = load_scalers(scaler_path)
    model2 = build_model_from_config(cfg_track2).to(device)
    ckpt   = torch.load(ckpt_path, map_location=device)
    model2.load_state_dict(ckpt["model_state"])
    model2.eval()

    # Charger quelques samples
    df_sample = df_val.sample(min(200, len(df_val)), random_state=42)
    vid_bbs   = cfg_track2.get("video", {}).get("active_backbones", [])
    val_loader2 = build_dataloader(
        df_sample,
        eeg_dir=Path("processed/features/eeg"),
        scalers=scalers,
        track=2,
        video_dir=Path("processed/features/video"),
        video_backbones=vid_bbs,
        batch_size=32,
        augment=False, shuffle=False,
    )

    gates_pos, gates_neg = [], []
    with torch.no_grad():
        for batch in val_loader2:
            eeg = batch["eeg"].to(device)
            vf  = {k: v.to(device) for k, v in batch.get("video_features", {}).items()}
            out = model2(eeg, video_features=vf or None, mode="track2")
            gate = out["gate"].cpu().numpy()
            lbl  = batch["label"].numpy()
            gates_pos.extend(gate[lbl == 1].tolist())
            gates_neg.extend(gate[lbl == 0].tolist())

    print(f"Gate moyenne (positifs) : {np.mean(gates_pos):.3f}")
    print(f"Gate moyenne (négatifs) : {np.mean(gates_neg):.3f}")
    print(f"(gate → 1 = modèle incertain, active la vidéo)")
else:
    print("Checkpoint Track 2 non trouvé — lancer l'entraînement d'abord.")

---
## 8. Track 3 — Localisation 5-classes

**Tâche** : sur les **positifs uniquement**, prédire la zone épileptogène (5 classes).  
**Métrique** : Weighted-F1 (classes 1-5, pondérées par fréquence réelle).  
**Spécificités** :
- Dataset filtré automatiquement sur `label==1`
- `label_type` 1-5 → remappé en 0-4 pour CrossEntropyLoss
- `WeightedCELoss` avec poids inverses à la fréquence (par fold)
- Channel-level features du GAT → `ChannelTemporalAttn` pour la localisation spatiale

In [ ]:
# ── Fine-tuning Track 3 ───────────────────────────────────────────────────
!python scripts/train_track.py \
    --track 3 \
    --config configs/track3.yaml \
    --backbone_ckpt {BACKBONE_CKPT}

In [ ]:
# ── Évaluation Track 3 : Weighted-F1 ─────────────────────────────────────
from src.evaluation.metrics import compute_weighted_f1

ckpt_path3   = Path("checkpoints/track3/fold0.pt")
scaler_path3 = Path("checkpoints/track3/scalers_fold0.pkl")

if ckpt_path3.exists() and scaler_path3.exists():
    scalers3 = load_scalers(scaler_path3)

    # Positifs uniquement en val
    df_val_pos = df_val[df_val["label"] == 1].reset_index(drop=True)
    print(f"Samples val positifs : {len(df_val_pos)}")

    vid_bbs3 = cfg_track3.get("video", {}).get("active_backbones", [])
    val_loader3 = build_dataloader(
        df_val_pos,
        eeg_dir=Path("processed/features/eeg"),
        scalers=scalers3,
        track=3,
        video_dir=Path("processed/features/video"),
        video_backbones=vid_bbs3,
        batch_size=64,
        augment=False, shuffle=False,
    )

    model3 = build_model_from_config(cfg_track3).to(device)
    ckpt3  = torch.load(ckpt_path3, map_location=device)
    model3.load_state_dict(ckpt3["model_state"])
    model3.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in val_loader3:
            eeg = batch["eeg"].to(device)
            vf  = {k: v.to(device) for k, v in batch.get("video_features", {}).items()}
            out = model3(eeg, video_features=vf or None, mode="track3")
            preds = torch.argmax(out["logit"], dim=-1).cpu().numpy() + 1  # → classes 1-5
            all_preds.extend(preds.tolist())
            all_targets.extend(batch["label_type"].numpy().tolist())

    wf1 = compute_weighted_f1(all_targets, all_preds)
    print(f"\nTrack 3 Weighted-F1 (fold 0) : {wf1:.4f}")

    # Distribution des prédictions vs cibles
    pred_counts  = pd.Series(all_preds).value_counts().sort_index()
    target_counts = pd.Series(all_targets).value_counts().sort_index()
    print("\nDistribution prédictions vs cibles :")
    comparison = pd.DataFrame({"Prédictions": pred_counts, "Cibles": target_counts})
    print(comparison.to_string())
else:
    print("Checkpoint Track 3 non trouvé — lancer l'entraînement d'abord.")

---
## 9. Évaluation & Calibration

### Calibration isotonique (Track 1/2)

L'`OOFCalibrator` applique une régression isotonique aux scores Out-of-Fold.  
**Règle absolue** : fitté uniquement sur les scores OOF — **jamais** sur le set val/candidat.

In [ ]:
from src.evaluation.metrics import OOFCalibrator, find_best_threshold

# ── Démonstration de la calibration ──────────────────────────────────────
# Simuler des scores OOF sur 3 folds
np.random.seed(42)
calib = OOFCalibrator()
all_t, all_s = [], []

for fold in range(5):
    n = 500
    t = np.random.randint(0, 2, n)
    # Scores imparfaitement calibrés (simulés)
    s = np.where(t == 1, np.random.beta(4, 2, n), np.random.beta(2, 4, n))
    calib.add_fold(t, s)
    all_t.extend(t.tolist())
    all_s.extend(s.tolist())

calib.fit()
all_t = np.array(all_t)
all_s = np.array(all_s)
cal_s = calib.transform(all_s)

from src.evaluation.metrics import compute_auprc
print(f"AUPRC avant calibration : {compute_auprc(all_t, all_s):.4f}")
print(f"AUPRC après calibration : {compute_auprc(all_t, cal_s):.4f}")

# Seuil optimal
thresh, f1 = find_best_threshold(all_t, cal_s)
print(f"\nSeuil optimal (F1-max) : {thresh:.3f}  →  F1 = {f1:.4f}")

In [ ]:
# ── Courbe de calibration (Reliability Diagram) ──────────────────────────
try:
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, scores, label, color in zip(
        axes,
        [all_s, cal_s],
        ["Avant calibration", "Après calibration isotonique"],
        ["steelblue", "crimson"],
    ):
        frac_pos, mean_pred = calibration_curve(all_t, scores, n_bins=10)
        ax.plot(mean_pred, frac_pos, "o-", color=color, label=label)
        ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Calibration parfaite")
        ax.set_xlabel("Score prédit moyen")
        ax.set_ylabel("Fraction de positifs")
        ax.set_title(label)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.suptitle("Reliability Diagram — Track 1/2", fontsize=13)
    plt.tight_layout()
    plt.savefig("logs/calibration_curve.png", dpi=100, bbox_inches="tight")
    plt.show()
except ImportError:
    print("matplotlib non installé — skip visualisation")

---
## 10. Inférence et soumission

Génère les fichiers CSV + ZIP pour Codabench.

**Règles critiques** :
- `batch_size=1` — aucune statistique globale sur le set candidat
- Scalers issus du **training complet** (pas d'un fold)
- Calibration isotonique appliquée si disponible
- Ensemble des 5 checkpoints par fold

In [ ]:
# ── Inférence Track 1 ─────────────────────────────────────────────────────
!python scripts/predict_candidate.py \
    --track 1 \
    --config configs/track1.yaml

In [ ]:
# ── Inférence Track 2 ─────────────────────────────────────────────────────
!python scripts/predict_candidate.py \
    --track 2 \
    --config configs/track2.yaml

In [ ]:
# ── Inférence Track 3 ─────────────────────────────────────────────────────
!python scripts/predict_candidate.py \
    --track 3 \
    --config configs/track3.yaml

In [ ]:
# ── Vérification des fichiers de soumission ───────────────────────────────
import zipfile

submissions_dir = Path("submissions")
print("=== Fichiers de soumission ===")

for track in [1, 2, 3]:
    csv_path = submissions_dir / f"track{track}_predictions.csv"
    zip_path = submissions_dir / f"track{track}_submission.zip"

    if csv_path.exists():
        df_sub = pd.read_csv(csv_path)
        col = "score" if track in (1, 2) else "prediction"
        print(f"\nTrack {track} :")
        print(f"  CSV  : {csv_path}  ({len(df_sub)} lignes)")
        print(f"  Colonnes : {list(df_sub.columns)}")
        if track in (1, 2):
            print(f"  Score min/max : {df_sub[col].min():.4f} / {df_sub[col].max():.4f}")
        else:
            print(f"  Distribution : {df_sub[col].value_counts().sort_index().to_dict()}")
        if zip_path.exists():
            print(f"  ZIP  : {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"\nTrack {track} : non généré")

---
## 11. Tests unitaires

La suite pytest couvre :
- RMSNorm, NativeMambaBlock, EEGMambaEncoder
- DynamicGAT (shape, gradients, residual)
- EEGBackbone (embed + channel_feats, freeze/unfreeze)
- NeuroMMModel (tous les modes + backward)
- Losses (FocalLoss, PolyLoss, FocalPolyLoss, WeightedCE, MultiTask)
- Metrics (AUPRC, Weighted-F1, OOFCalibrator, MetricTracker)
- Augmentations (SpecAugment, GaussianNoise, AugmentationPipeline)
- Config (load_config, build_model_from_config)

In [ ]:
# ── Suite complète ────────────────────────────────────────────────────────
!python -m pytest tests/ -v --tb=short 2>&1 | tail -50

In [ ]:
# ── Tests rapides inline (sans dépendances fichiers) ─────────────────────
import torch
import numpy as np
from src.models.backbone import EEGBackbone, BackboneConfig
from src.models.eegmamba_encoder import EEGMambaEncoder, NativeMambaBlock
from src.models.dynamic_gat import DynamicGAT
from src.training.losses import FocalPolyLoss, MultiTaskLoss
from src.evaluation.metrics import compute_auprc, OOFCalibrator
from src.data.augmentations import AugmentationPipeline

B, C, T = 4, 29, 2000
results = {}

# 1. NativeMambaBlock
block = NativeMambaBlock(d_model=64, d_state=8, d_conv=4, expand=2)
x = torch.randn(B, C, 64)
assert block(x).shape == (B, C, 64), "MambaBlock shape FAIL"
results["NativeMambaBlock"] = "PASS"

# 2. EEGMambaEncoder
enc = EEGMambaEncoder(d_model=64, n_layers=2, force_native=True)
assert enc(torch.randn(B, C, 64)).shape == (B, 64), "Encoder shape FAIL"
results["EEGMambaEncoder"] = "PASS"

# 3. DynamicGAT
gat = DynamicGAT(in_dim=32, hidden_dim=64, n_layers=2, n_heads=4)
assert gat(torch.randn(B, C, 32)).shape == (B, C, 64), "GAT shape FAIL"
results["DynamicGAT"] = "PASS"

# 4. Backbone complet
cfg = BackboneConfig(cnn_out_channels=32, gat_hidden_dim=64, mamba_d_model=64,
                     mamba_d_state=8, mamba_n_layers=2, embed_dim=64)
bb = EEGBackbone(cfg)
embed = bb(torch.randn(B, C, T))
assert embed.shape == (B, 64), "Backbone embed shape FAIL"
embed2, ch = bb(torch.randn(B, C, T), return_channel_feats=True)
assert ch.shape == (B, C, 64), "Backbone channel_feats FAIL"
results["EEGBackbone"] = "PASS"

# 5. FocalPolyLoss
loss_fn = FocalPolyLoss()
logits  = torch.randn(32)
targets = torch.randint(0, 2, (32,)).float()
loss = loss_fn(logits, targets)
assert loss.item() > 0 and not torch.isnan(loss), "FocalPolyLoss FAIL"
loss.backward()
results["FocalPolyLoss"] = "PASS"

# 6. AUPRC
t = np.array([0, 0, 1, 1, 0, 1])
s = np.array([0.1, 0.2, 0.8, 0.9, 0.3, 0.7])
assert compute_auprc(t, s) > 0.9, "AUPRC FAIL"
results["compute_auprc"] = "PASS"

# 7. OOFCalibrator
calib = OOFCalibrator()
for _ in range(3):
    calib.add_fold(np.random.randint(0, 2, 50), np.random.rand(50))
calib.fit()
out = calib.transform(np.array([0.2, 0.5, 0.8]))
assert out.shape == (3,), "OOFCalibrator FAIL"
results["OOFCalibrator"] = "PASS"

# 8. Augmentations
aug = AugmentationPipeline({
    "spec_augment":   {"enabled": True, "max_time_mask": 100, "num_time_masks": 1},
    "gaussian_noise": {"enabled": True, "sigma": 0.05},
})
eeg_np = np.random.randn(29, 2000).astype(np.float32)
assert aug(eeg_np).shape == (29, 2000), "Augmentation FAIL"
results["AugmentationPipeline"] = "PASS"

# Résumé
print("=== Résultats des tests inline ===")
all_pass = True
for name, status in results.items():
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon}  {name:<30} {status}")
    if status != "PASS":
        all_pass = False

print(f"\n{'✓ Tous les tests passent !' if all_pass else '✗ Des tests ont échoué.'}")

---
## Récapitulatif du pipeline

```
prepare_data.py           →  annotations/ + processed/features/
       ↓
pretrain.py               →  checkpoints/pretrain/best_backbone.pt
       ↓
train_track.py --track 1  →  checkpoints/track1/fold{0..4}.pt + calibrator.pkl
train_track.py --track 2  →  checkpoints/track2/fold{0..4}.pt + calibrator.pkl
train_track.py --track 3  →  checkpoints/track3/fold{0..4}.pt
       ↓
predict_candidate.py      →  submissions/track{1,2,3}_submission.zip
       ↓
Codabench upload          →  https://2026.neuromm.org
```

### Anti-leakage checklist
- [x] `GroupKFold(subject_id)` — patient-disjoint
- [x] `RobustScaler` fitté sur fold-train uniquement
- [x] Calibration isotonique sur OOF uniquement
- [x] `batch_size=1` pour l'inférence candidat
- [x] Aucune normalisation globale sur le set candidat

### Références
- Mamba SSM : [Gu & Dao, 2023](https://arxiv.org/abs/2312.00752)
- FocalLoss : [Lin et al., 2017](https://arxiv.org/abs/1708.02002)
- PolyLoss  : [Leng et al., 2022](https://arxiv.org/abs/2204.12511)
- Dataset   : https://huggingface.co/datasets/NeuroMM/NeuroMM-2026
- Leaderboard: https://2026.neuromm.org